# ML Experiment Orchestrator — Exploration Notebook

Use this notebook to interactively explore your data and prototype ideas
before wiring them into the `src/` pipeline modules.

> **Note:** All importable logic lives in `src/`. Keep this notebook as a
> thin wrapper around those modules so the codebase stays testable.

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure the project root is on sys.path so `src` is importable
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

In [ ]:
# ── Import orchestrator modules ───────────────────────────────────────────
from src.utils.config import load_config
from src.utils.logger import get_logger
from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.data.splitter import DataSplitter
from src.models.trainer import ModelTrainer
from src.models.evaluator import Evaluator

log = get_logger('notebook')

In [ ]:
# ── Load configuration ────────────────────────────────────────────────────
cfg = load_config('../configs/experiment.yaml')
cfg

In [ ]:
# ── Load data (generates a dummy dataset if none exists) ──────────────────
from pathlib import Path
import numpy as np

filepath = cfg['data']['filepath']
if not Path(filepath).exists():
    rng = np.random.default_rng(42)
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    dummy = pd.DataFrame({
        'feature_1': rng.standard_normal(200),
        'feature_2': rng.standard_normal(200),
        'feature_3': rng.uniform(0, 10, 200),
        'target':    rng.integers(0, 2, 200),
    })
    dummy.to_csv(filepath, index=False)
    print(f'Generated dummy data at: {filepath}')

loader = DataLoader()
df = loader.load(filepath)
df.head()

In [ ]:
# ── Quick EDA ─────────────────────────────────────────────────────────────
print(df.describe())
print('\nNull counts:')
print(df.isnull().sum())

In [ ]:
# ── Clean → Split → Train → Evaluate ─────────────────────────────────────
cleaner  = DataCleaner(cfg['cleaning'])
df_clean = cleaner.clean(df)

splitter = DataSplitter(**cfg['splitting'])
splits   = splitter.split(df_clean, cfg['data']['target_col'])

trainer  = ModelTrainer(cfg['model']['type'], cfg['model'].get('params', {}))
trainer.fit(splits['X_train'], splits['y_train'])

evaluator = Evaluator()
metrics   = evaluator.evaluate(trainer.model, splits['X_test'], splits['y_test'])
metrics